# Download GDPval Task Data

Downloads the 3 target tasks from the [GDPval HuggingFace dataset](https://huggingface.co/datasets/openai/gdpval).

**Data integrity note**: This notebook downloads ONLY task prompts and reference files.  
It deliberately skips deliverable files and rubrics to avoid data leakage in benchmarking.

## Target tasks
- Fall Music Tour P&L
- Aurisic Prepaid Amortization  
- Anti-Financial Crime Audit Sampling

All three are from: `occupation = Accountants and Auditors`

In [1]:
# Install dependencies if needed
!pip install datasets requests

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.4.1-cp311-cp311-win_amd64.whl.metadata (14 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 4.3 MB/s  0:00:00
   ---------------------------------------- 0.0/625.2 kB ? eta -:--:--
   --------------------------------- ------ 524.3/625.2 kB ? eta -:--:--
   -----------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset
import requests
import os
from pathlib import Path

# Load the dataset (public, no auth required)
print("Loading GDPval dataset...")
ds = load_dataset("openai/gdpval", split="train")
print(f"Loaded {len(ds)} tasks")

c:\Users\York Yong\.conda\envs\ClearMyBoss\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GDPval dataset...


Generating train split: 100%|██████████| 220/220 [00:00<00:00, 6666.43 examples/s]

Loaded 220 tasks


In [3]:
# Inspect available fields
print("Fields:", ds.column_names)
print()

# Show all Accountants and Auditors tasks
accountant_tasks = [row for row in ds if row["occupation"] == "Accountants and Auditors"]
print(f"Found {len(accountant_tasks)} Accountants and Auditors tasks:")
for i, row in enumerate(accountant_tasks):
    print(f"\n--- Task {i+1} ---")
    print(f"task_id: {row['task_id']}")
    # Show first 200 chars of prompt as a preview
    print(f"prompt preview: {row['prompt'][:200]}...")
    print(f"reference_files: {row['reference_files']}")

Fields: ['task_id', 'sector', 'occupation', 'prompt', 'reference_files', 'reference_file_urls', 'reference_file_hf_uris', 'deliverable_files', 'deliverable_file_urls', 'deliverable_file_hf_uris', 'rubric_pretty', 'rubric_json']

Found 5 Accountants and Auditors tasks:

--- Task 1 ---
task_id: 83d10b06-26d1-4636-a32c-23f92c57f30b
prompt preview: You are an auditor and as part of an audit engagement, you are tasked with reviewing and testing the accuracy of reported Anti-Financial Crime Risk Metrics.

The attached spreadsheet titled ‘Populatio...
reference_files: ['reference_files/cc781e4dc0985c8eb327a53ec03b5900/Population v2.xlsx']

--- Task 2 ---
task_id: 7b08cd4d-df60-41ae-9102-8aaa49306ba2
prompt preview: You are the Finance Lead for an advisory client and are responsible for managing and controlling expenses related to their professional music engagements. Your summary will be used not only for intern...
reference_files: ['reference_files/4e6e2b8d17f751e483aad52c109813b4/Fall Music

In [4]:
# --- IDENTIFY THE 3 TARGET TASKS ---
# Search by keyword in the prompt field.
# Adjust the keywords below if the exact match doesn't work.

TASK_KEYWORDS = {
    "music-tour-pl": ["music tour", "P&L", "profit and loss", "tour"],
    "aurisic-amortization": ["aurisic", "amortization", "prepaid"],
    "afc-audit-sampling": ["anti-financial crime", "AFC", "audit sampling", "financial crime risk"]
}

found_tasks = {}

for task_key, keywords in TASK_KEYWORDS.items():
    for row in ds:
        prompt_lower = row["prompt"].lower()
        if any(kw.lower() in prompt_lower for kw in keywords):
            if task_key not in found_tasks:
                found_tasks[task_key] = row
                print(f"FOUND [{task_key}]: task_id={row['task_id']}")
                print(f"  Occupation: {row['occupation']}")
                print(f"  Reference files: {row['reference_files']}")
                print(f"  Prompt preview: {row['prompt'][:300]}...")
                print()
                break

missing = [k for k in TASK_KEYWORDS if k not in found_tasks]
if missing:
    print(f"WARNING: Could not find tasks: {missing}")
    print("Try printing all accountant_tasks above and manually identify the correct task_ids")
else:
    print("All 3 target tasks found. Proceed to download.")

FOUND [music-tour-pl]: task_id=7b08cd4d-df60-41ae-9102-8aaa49306ba2
  Occupation: Accountants and Auditors
  Reference files: ['reference_files/4e6e2b8d17f751e483aad52c109813b4/Fall Music Tour Ref File.xlsx']
  Prompt preview: You are the Finance Lead for an advisory client and are responsible for managing and controlling expenses related to their professional music engagements. Your summary will be used not only for internal oversight but also by executives at the production company to evaluate tour performance and guide...

FOUND [aurisic-amortization]: task_id=7d7fc9a7-21a7-4b83-906f-416dea5ad04f
  Occupation: Accountants and Auditors
  Reference files: ['reference_files/6498264b7ee431a71a604675222584eb/COA.xlsx', 'reference_files/2f0f77ed28ec98110006c77c286558fc/Aurisic_Prepaid_Expenses_Apr25.pdf', 'reference_files/7ed8b041310d72169ceb6595819b84a0/Aurisic_Prepaid_Expenses_Mar25.pdf', 'reference_files/0d96c101001bcad1d8cc0c2d6de5df74/Aurisic_Prepaid_Expenses_Feb25.pdf', 'reference_f

In [5]:
# --- REVIEW BEFORE DOWNLOADING ---
# Confirm the found tasks look correct before proceeding.
# If a task was misidentified, manually set the task_id here:

# Example override (uncomment and set task_id if needed):
# found_tasks["music-tour-pl"] = next(r for r in ds if r["task_id"] == "<PASTE_TASK_ID_HERE>")

print("Tasks to be downloaded:")
for key, row in found_tasks.items():
    print(f"  {key}: {row['task_id']} ({len(row['reference_files'])} reference files)")

Tasks to be downloaded:
  music-tour-pl: 7b08cd4d-df60-41ae-9102-8aaa49306ba2 (1 reference files)
  aurisic-amortization: 7d7fc9a7-21a7-4b83-906f-416dea5ad04f (6 reference files)
  afc-audit-sampling: 83d10b06-26d1-4636-a32c-23f92c57f30b (1 reference files)


In [6]:
# --- DOWNLOAD REFERENCE FILES AND WRITE PROMPT.MD ---
# DATA INTEGRITY: Only reference files are downloaded. Deliverable files and rubrics are skipped.

BASE_DIR = Path(".")  # This notebook is at benchmarks/tasks/

TASK_DIRS = {
    "music-tour-pl": BASE_DIR / "music-tour-pl",
    "aurisic-amortization": BASE_DIR / "aurisic-amortization",
    "afc-audit-sampling": BASE_DIR / "afc-audit-sampling",
}

def download_file(url: str, dest_path: Path):
    """Download a file from a URL to a local path."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"  Downloaded: {dest_path.name}")


for task_key, row in found_tasks.items():
    task_dir = TASK_DIRS[task_key]
    ref_dir = task_dir / "reference-files"
    ref_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing [{task_key}]...")

    # 1. Write prompt.md
    prompt_path = task_dir / "prompt.md"
    prompt_content = f"""# Task Prompt: {task_key}

> Source: GDPval benchmark, OpenAI (2025). Accountants & Auditors occupation, Professional/Scientific/Technical Services sector.
> task_id: {row['task_id']}
> Reference: https://huggingface.co/datasets/openai/gdpval

---

{row['prompt']}
"""
    prompt_path.write_text(prompt_content, encoding="utf-8")
    print(f"  Wrote prompt.md")

    # 2. Download reference files ONLY (skip deliverable_files and rubric)
    ref_urls = row.get("reference_file_urls", [])
    ref_names = row.get("reference_files", [])

    if not ref_urls:
        print(f"  No reference files for this task.")
        continue

    for url, filepath in zip(ref_urls, ref_names):
        filename = Path(filepath).name
        dest = ref_dir / filename
        try:
            download_file(url, dest)
        except Exception as e:
            print(f"  ERROR downloading {filename}: {e}")

print("\nDone. Reference files saved. Rubric and deliverable files were NOT downloaded.")


Processing [music-tour-pl]...
  Wrote prompt.md
  Downloaded: Fall Music Tour Ref File.xlsx

Processing [aurisic-amortization]...
  Wrote prompt.md
  Downloaded: COA.xlsx
  Downloaded: Aurisic_Prepaid_Expenses_Apr25.pdf
  Downloaded: Aurisic_Prepaid_Expenses_Mar25.pdf
  Downloaded: Aurisic_Prepaid_Expenses_Feb25.pdf
  Downloaded: Aurisic_Prepaid_Expenses_Jan25.pdf
  Downloaded: Aurisic_Prepaid_Insurance.pdf

Processing [afc-audit-sampling]...
  Wrote prompt.md
  Downloaded: Population v2.xlsx

Done. Reference files saved. Rubric and deliverable files were NOT downloaded.


In [7]:
# --- VERIFY DOWNLOADS ---
print("Download summary:\n")
for task_key, task_dir in TASK_DIRS.items():
    ref_dir = task_dir / "reference-files"
    files = list(ref_dir.iterdir()) if ref_dir.exists() else []
    prompt_exists = (task_dir / "prompt.md").exists()
    print(f"{task_key}:")
    print(f"  prompt.md: {'OK' if prompt_exists else 'MISSING'}")
    print(f"  reference files ({len(files)}): {[f.name for f in files]}")

Download summary:

music-tour-pl:
  prompt.md: OK
  reference files (1): ['Fall Music Tour Ref File.xlsx']
aurisic-amortization:
  prompt.md: OK
  reference files (6): ['Aurisic_Prepaid_Expenses_Apr25.pdf', 'Aurisic_Prepaid_Expenses_Feb25.pdf', 'Aurisic_Prepaid_Expenses_Jan25.pdf', 'Aurisic_Prepaid_Expenses_Mar25.pdf', 'Aurisic_Prepaid_Insurance.pdf', 'COA.xlsx']
afc-audit-sampling:
  prompt.md: OK
  reference files (1): ['Population v2.xlsx']
